# Bluestock Mutual Fund Analytics - Day 3: Performance & Risk Scorecard

This notebook implements the performance analytics, risk metrics calculation, regression analysis against benchmark indices, and the final fund scorecard ranking for the six mutual funds in the repository.

## Data Sources & Documentation
1. **Historical Mutual Fund NAVs**: Cleaned daily NAV values from `data/processed/nav_history_cleaned.csv` (originating from AMFI).
2. **Benchmark index data**: Historical daily prices for Nifty 50 (`^NSEI`) and Nifty 100 (`^CNX100`) fetched from the public **Yahoo Finance Chart API** (covering 2012-12-31 to 2026-06-19).
3. **Fund Metadata (AUM & Expense Ratios)**: Verified from public disclosures on **ValueResearchOnline** as of June 2026 and loaded from `data/raw/fund_metadata.csv`.
   - *Nippon India Large Cap (118632)*: AUM ₹51,660.30 Cr | Expense 0.58% | Benchmark: Nifty 50
   - *HDFC Money Market (119092)*: AUM ₹28,705.00 Cr | Expense 0.22% | Benchmark: Nifty 50
   - *ABSL Banking & PSU (119551)*: AUM ₹8,820.00 Cr | Expense 0.33% | Benchmark: Nifty 50
   - *Axis ELSS Tax Saver (120503)*: AUM ₹31,023.00 Cr | Expense 0.87% | Benchmark: Nifty 50
   - *quant Mid Cap (120841)*: AUM ₹8,109.12 Cr | Expense 0.74% | Benchmark: Nifty 100
   - *SBI Small Cap (125497)*: AUM ₹37,395.00 Cr | Expense 0.74% | Benchmark: Nifty 100


### 1. Setup and Library Imports
We import the necessary libraries for calculations, statistical analysis, linear regression, and visualizations.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress, skew, kurtosis
from datetime import datetime

# Set plot aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Risk-free rate (6.5% per annum)
RF_ANNUAL = 0.065
RF_DAILY = RF_ANNUAL / 252
print("Setup complete. Daily risk-free rate:", RF_DAILY)

### 2. Load the Data
We load the clean daily mutual fund NAVs, the verified AUM/expense ratio metadata, and the benchmark returns.

In [ ]:
nav_df = pd.read_csv('../data/processed/nav_history_cleaned.csv')
metadata_df = pd.read_csv('../data/raw/fund_metadata.csv')
bench_df = pd.read_csv('../data/processed/nifty_benchmarks_cleaned.csv')

nav_df['date'] = pd.to_datetime(nav_df['date']).dt.strftime('%Y-%m-%d')
bench_df['date'] = pd.to_datetime(bench_df['date']).dt.strftime('%Y-%m-%d')

print(f"Loaded {len(nav_df)} NAV records, {len(metadata_df)} metadata schemes, and {len(bench_df)} benchmark records.")

### 3. Data Inconsistency Correction
HDFC Money Market Fund (`119092`) has a known 100x scaling error in its AMFI historical NAV entries before `2015-08-30` (where NAV was around 30 vs 3000 after). We apply a 100x correction to resolve this decimal placement typo.

In [ ]:
# Sort data first
nav_df = nav_df.sort_values(['amfi_code', 'date']).reset_index(drop=True)

# Apply HDFC Money Market NAV scaling correction
nav_df['nav'] = np.where(
    (nav_df['amfi_code'] == 119092) & (nav_df['date'] < '2015-08-30'),
    nav_df['nav'] * 100.0,
    nav_df['nav']
)
print("Outlier correction applied successfully.")

### 4. Daily Returns and Distribution Validation
We calculate the daily returns for each fund: $R_t = \frac{NAV_t}{NAV_{t-1}} - 1$.
We validate their distribution by printing the mean, standard deviation, skewness, and kurtosis.

In [ ]:
nav_df['daily_return'] = nav_df.groupby('amfi_code')['nav'].pct_change()

# Validate distributions
grouped = nav_df.groupby('amfi_code')
for amfi_code, df_fund in grouped:
    returns = df_fund['daily_return'].dropna()
    scheme_name = df_fund.iloc[0]['scheme_name']
    print(f"{scheme_name:65} | Mean={returns.mean()*100:8.4f}% | Std={returns.std()*100:8.4f}% | Skew={skew(returns):8.4f} | Kurt={kurtosis(returns):8.4f}")

### 5. CAGR Calculation (1Y, 3Y, 5Y)
We compute CAGR using the closest date logic. For weekends/holidays, we select the closest trading date within a 7-day window.
Formula: $\text{CAGR} = \left(\frac{\text{NAV}_{\text{end}}}{\text{NAV}_{\text{start}}}\right)^{\frac{1}{\text{years}}} - 1$.

In [ ]:
def get_nav_on_date(df_fund, target_date_str):
    df = df_fund.copy()
    target_dt = pd.to_datetime(target_date_str)
    df['date_dt'] = pd.to_datetime(df['date'])
    diff = (df['date_dt'] - target_dt).abs()
    closest_idx = diff.idxmin()
    if diff.loc[closest_idx].days <= 7:
        return df.loc[closest_idx, 'nav'], df.loc[closest_idx, 'date']
    return None, None

def calculate_cagr(nav_start, nav_end, years):
    if nav_start is None or nav_end is None or nav_start <= 0 or nav_end <= 0:
        return np.nan
    return (nav_end / nav_start) ** (1.0 / years) - 1.0

latest_date_str = "2026-06-19"
cagr_records = []

for amfi_code, df_fund in grouped:
    df_fund = df_fund.dropna(subset=['nav']).sort_values('date').reset_index(drop=True)
    name = df_fund.iloc[0]['scheme_name']
    
    nav_latest, date_latest = get_nav_on_date(df_fund, latest_date_str)
    nav_1y, date_1y = get_nav_on_date(df_fund, "2025-06-19")
    nav_3y, date_3y = get_nav_on_date(df_fund, "2023-06-19")
    nav_5y, date_5y = get_nav_on_date(df_fund, "2021-06-19")
    
    c1 = calculate_cagr(nav_1y, nav_latest, 1.0)
    c3 = calculate_cagr(nav_3y, nav_latest, 3.0)
    c5 = calculate_cagr(nav_5y, nav_latest, 5.0)
    
    cagr_records.append({
        'amfi_code': amfi_code,
        'scheme_name': name,
        'cagr_1y': c1,
        'cagr_3y': c3,
        'cagr_5y': c5
    })
    
cagr_df = pd.DataFrame(cagr_records)
print(cagr_df[['scheme_name', 'cagr_1y', 'cagr_3y', 'cagr_5y']])

### 6. Risk-Adjusted Returns: Sharpe and Sortino Ratios
We calculate the Sharpe and Sortino ratios using the 6.5% risk-free rate.
Sharpe is annualized: $\text{Sharpe} = \frac{\text{AnnReturn} - R_f}{\text{AnnVol}}$.
Sortino is based on downside deviation: $\text{Sortino} = \frac{\text{AnnReturn} - R_f}{\text{DownsideVol}}$.

In [ ]:
def calculate_downside_deviation(returns, target_rate=RF_DAILY):
    excess_returns = returns - target_rate
    downside_returns = np.minimum(0, excess_returns)
    downside_var = np.mean(downside_returns ** 2)
    return np.sqrt(downside_var) * np.sqrt(252)

risk_records = []

for amfi_code, df_fund in grouped:
    df_fund = df_fund.dropna(subset=['daily_return'])
    name = df_fund.iloc[0]['scheme_name']
    rets = df_fund['daily_return']
    
    ann_ret = rets.mean() * 252
    ann_vol = rets.std() * np.sqrt(252)
    
    sharpe = (ann_ret - RF_ANNUAL) / ann_vol if ann_vol > 0 else np.nan
    downside_dev = calculate_downside_deviation(rets)
    sortino = (ann_ret - RF_ANNUAL) / downside_dev if downside_dev > 0 else np.nan
    
    risk_records.append({
        'amfi_code': amfi_code,
        'sharpe_ratio': sharpe,
        'sortino_ratio': sortino,
        'volatility_annualized': ann_vol
    })
    
risk_df = pd.DataFrame(risk_records)
print(pd.merge(cagr_df[['amfi_code', 'scheme_name']], risk_df, on='amfi_code'))

### 7. Maximum Drawdown and Worst Drawdown Periods
We calculate the maximum peak-to-trough decline for each fund, identifying the peak, trough (maximum decline), and recovery dates.

In [ ]:
def calculate_drawdown_periods(df_fund):
    df = df_fund.copy().sort_values('date').reset_index(drop=True)
    df['peak'] = df['nav'].cummax()
    df['drawdown'] = (df['nav'] - df['peak']) / df['peak']
    
    max_dd = df['drawdown'].min()
    if pd.isna(max_dd) or max_dd == 0:
        return 0.0, "N/A", "N/A", "N/A"
        
    trough_idx = df['drawdown'].idxmin()
    trough_date = df.loc[trough_idx, 'date']
    
    peak_idx = df.loc[:trough_idx, 'nav'].idxmax()
    peak_date = df.loc[peak_idx, 'date']
    
    recovery_df = df.loc[trough_idx:]
    recovered_runs = recovery_df[recovery_df['nav'] >= df.loc[peak_idx, 'nav']]
    if not recovered_runs.empty:
        recovery_idx = recovered_runs.index[0]
        recovery_date = df.loc[recovery_idx, 'date']
    else:
        recovery_date = "Unrecovered"
        
    return max_dd, peak_date, trough_date, recovery_date

dd_records = []
for amfi_code, df_fund in grouped:
    df_fund = df_fund.dropna(subset=['nav'])
    max_dd, peak_date, trough_date, recovery_date = calculate_drawdown_periods(df_fund)
    dd_records.append({
        'amfi_code': amfi_code,
        'max_drawdown': max_dd,
        'worst_dd_peak_date': peak_date,
        'worst_dd_trough_date': trough_date,
        'worst_dd_recovery_date': recovery_date
    })

dd_df = pd.DataFrame(dd_records)
print(dd_df)

### 8. Regression against Benchmark Indices: Alpha, Beta, and Tracking Error
We align the dates between the fund daily returns and benchmark daily returns, drop missing values, and run a linear regression to compute Beta, Annualized Alpha, and Tracking Error.

In [ ]:
regression_records = []

for amfi_code, df_fund in grouped:
    df_fund = df_fund.dropna(subset=['nav'])
    name = df_fund.iloc[0]['scheme_name']
    
    # Determine benchmark ticker
    if "quant Mid" in name or "SBI Small" in name:
        bench_ticker = "^CNX100"
    else:
        bench_ticker = "^NSEI"
        
    fund_ret_df = df_fund[['date', 'daily_return']].dropna()
    merged_bench = pd.merge(fund_ret_df, bench_df[['date', f'{bench_ticker}_return']], on='date', how='inner').dropna()
    
    if not merged_bench.empty:
        bench_ret_series = merged_bench[f'{bench_ticker}_return']
        fund_ret_series = merged_bench['daily_return']
        
        slope, intercept, r_value, p_value, std_err = linregress(bench_ret_series, fund_ret_series)
        beta = slope
        daily_alpha = intercept
        ann_alpha = daily_alpha * 252
        
        tracking_error = np.std(fund_ret_series - bench_ret_series) * np.sqrt(252)
        
        regression_records.append({
            'amfi_code': amfi_code,
            'benchmark': bench_ticker,
            'alpha_annualized': ann_alpha,
            'beta': beta,
            'tracking_error_annualized': tracking_error
        })
        
reg_df = pd.DataFrame(regression_records)
print(reg_df)

### 9. Dynamic Fund Scorecard Ranking (0-100)
We construct a scorecard combining 3-Year CAGR, Sharpe, Sortino, Max Drawdown, and the Expense Ratio. If any metric is marked as `"Unavailable"`, weights are dynamically rescaled to ensure data integrity.

In [ ]:
# Merge all metrics
score_df = pd.merge(cagr_df, risk_df, on='amfi_code')
score_df = pd.merge(score_df, dd_df, on='amfi_code')
score_df = pd.merge(score_df, metadata_df[['scheme_code', 'aum_cr', 'expense_ratio_percent']], left_on='amfi_code', right_on='scheme_code', how='left')

# Min-max scaling
def scale_higher_better(series):
    s_min, s_max = series.min(), series.max()
    return (series - s_min) / (s_max - s_min) * 100.0 if s_max != s_min else pd.Series(100.0, index=series.index)

def scale_lower_better(series):
    s_min, s_max = series.min(), series.max()
    return (s_max - series) / (s_max - s_min) * 100.0 if s_max != s_min else pd.Series(100.0, index=series.index)

score_df['score_cagr_3y'] = scale_higher_better(score_df['cagr_3y'])
score_df['score_sharpe'] = scale_higher_better(score_df['sharpe_ratio'])
score_df['score_sortino'] = scale_higher_better(score_df['sortino_ratio'])
score_df['score_drawdown'] = scale_higher_better(score_df['max_drawdown'])
score_df['score_expense'] = scale_lower_better(score_df['expense_ratio_percent'])

base_weights = {
    'score_cagr_3y': 0.30,
    'score_sharpe': 0.25,
    'score_sortino': 0.20,
    'score_drawdown': 0.15,
    'score_expense': 0.10
}

final_scores = []
audit_logs = []

for idx, row in score_df.iterrows():
    total_weight = 0.0
    weighted_sum = 0.0
    omitted = []
    
    for k, score_col in [('score_cagr_3y', 'cagr_3y'), 
                         ('score_sharpe', 'sharpe_ratio'), 
                         ('score_sortino', 'sortino_ratio'), 
                         ('score_drawdown', 'max_drawdown'), 
                         ('score_expense', 'expense_ratio_percent')]:
        if not pd.isna(row[score_col]):
            total_weight += base_weights[k]
            weighted_sum += row[k] * base_weights[k]
        else:
            omitted.append(score_col)
            
    final_scores.append(round(weighted_sum / total_weight, 2) if total_weight > 0 else np.nan)
    audit_logs.append(", ".join(omitted) if omitted else "None")
    
score_df['overall_scorecard_score'] = final_scores
score_df['omitted_metrics'] = audit_logs

# Display final ranking
score_df = score_df.sort_values('overall_scorecard_score', ascending=False).reset_index(drop=True)
print(score_df[['scheme_name', 'cagr_3y', 'sharpe_ratio', 'max_drawdown', 'expense_ratio_percent', 'overall_scorecard_score']])

### 10. Visualizations: Cumulative returns comparison
We plot the cumulative growth of a normalized base of 100 for three core equity funds against the Nifty 50 and Nifty 100 benchmarks.

In [ ]:
plt.figure(figsize=(14, 8))
colors = {
    118632: '#1f77b4',  # Nippon India Large Cap
    120841: '#d62728',  # quant Mid Cap
    125497: '#2ca02c',  # SBI Small Cap
    '^NSEI': '#ff7f0e', # Nifty 50
    '^CNX100': '#9467bd' # Nifty 100
}

# Plot benchmarks
bench_plot_df = bench_df[bench_df['date'] >= '2013-01-02'].sort_values('date').reset_index(drop=True)
for ticker, label in [('^NSEI', 'Nifty 50 Index'), ('^CNX100', 'Nifty 100 Index')]:
    first_val = bench_plot_df[f'{ticker}_close'].iloc[0]
    plt.plot(pd.to_datetime(bench_plot_df['date']), (bench_plot_df[f'{ticker}_close'] / first_val) * 100.0,
             label=label, color=colors[ticker], linewidth=1.5, linestyle='--')

# Plot funds
for code in [118632, 120841, 125497]:
    df_f = nav_df[(nav_df['amfi_code'] == code) & (nav_df['date'] >= '2013-01-02')].sort_values('date').reset_index(drop=True)
    first_nav = df_f['nav'].iloc[0]
    name = df_f.iloc[0]['scheme_name'].split('-')[0].strip()
    plt.plot(pd.to_datetime(df_f['date']), (df_f['nav'] / first_nav) * 100.0,
             label=name, color=colors[code], linewidth=2.0)

plt.title("Normalized Cumulative Growth Comparison (2013 - 2026) | Base = 100", fontsize=15, fontweight='bold', pad=15)
plt.xlabel("Year", fontsize=12, labelpad=10)
plt.ylabel("Cumulative Value", fontsize=12, labelpad=10)
plt.legend(loc='upper left', frameon=True, fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()